## ¿Qué vamos a hacer?

Vamos a enseñarle a una computadora a reconocer dígitos escritos a mano (0-9) usando una **red neuronal implementada con PyTorch**. Usaremos el dataset **MNIST**, que contiene 70,000 imágenes de dígitos de 28×28 píxeles.

Este notebook está preparado para utilizar una **GPU en Google Colab** cuando esté disponible.

> En Colab: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets

sns.set_theme(style='whitegrid', palette='muted')

# Semillas para que los resultados sean más reproducibles.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
print("Versión de PyTorch:", torch.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo seleccionado:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No se detectó una GPU. En Colab, activa T4 GPU desde el menú de entorno de ejecución.")

In [ ]:
# torchvision descarga MNIST la primera vez que se ejecuta.
train_dataset_raw = datasets.MNIST(root="./data", train=True, download=True)
test_dataset_raw = datasets.MNIST(root="./data", train=False, download=True)

# Los convertimos temporalmente a NumPy para conservar las visualizaciones
# didácticas de las siguientes celdas.
train_images = train_dataset_raw.data.numpy()
train_labels = train_dataset_raw.targets.numpy()

test_images = test_dataset_raw.data.numpy()
test_labels = test_dataset_raw.targets.numpy()

## Conociendo los datos

Ya cargamos el dataset. Veamos qué contiene: 60,000 imágenes para entrenar y 10,000 para probar. Cada imagen es una cuadrícula de 28×28 píxeles.

In [ ]:
train_images.shape

In [ ]:
len([1, 4, 6, 6])

In [ ]:
len(train_labels)

In [ ]:
train_labels

In [ ]:
test_images.shape

In [ ]:
len(test_labels)

In [ ]:
test_labels

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(15, 4))
fig.suptitle('Ejemplos de dígitos en el dataset', fontsize=14, fontweight='bold')

for digito in range(10):
    indices = np.where(train_labels == digito)[0]
    for fila in range(2):
        ax = axes[fila, digito]
        ax.imshow(train_images[indices[fila]], cmap='gray')
        ax.set_title(str(digito), fontsize=12)
        ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
ejemplo_idx = np.where(train_labels == 5)[0][0]
ejemplo = train_images[ejemplo_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.imshow(ejemplo, cmap='gray')
ax1.set_title(f'Así lo vemos nosotros (dígito: {train_labels[ejemplo_idx]})', fontsize=12)
ax1.axis('off')

zona = ejemplo[6:20, 6:20]
sns.heatmap(zona, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax2,
            cbar_kws={'label': 'Valor del píxel (0-255)'},
            linewidths=0.5, linecolor='white')
ax2.set_title('Así lo ve la computadora (zona central 14×14)', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
digitos, conteos = np.unique(train_labels, return_counts=True)
bars = ax.bar(digitos, conteos, color=sns.color_palette('muted', 10), edgecolor='black')

for bar, conteo in zip(bars, conteos):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100,
            f'{conteo:,}', ha='center', va='bottom', fontsize=10)

ax.set_xlabel('Dígito', fontsize=12)
ax.set_ylabel('Cantidad de imágenes', fontsize=12)
ax.set_title('¿Cuántas imágenes hay de cada dígito?', fontsize=14, fontweight='bold')
ax.set_xticks(range(10))
plt.tight_layout()
plt.show()

## Construyendo la red neuronal

Vamos a definir la estructura de nuestra red: cuántas capas tiene, cuántas neuronas por capa, y qué función de activación usa cada una.

In [ ]:
# nn.Sequential conecta las capas en el orden en que deben ejecutarse.
network = nn.Sequential(
    nn.Linear(28 * 28, 32),
    nn.Tanh(),
    nn.Linear(32, 16),
    nn.Sigmoid(),
    nn.Linear(16, 10)
)

# Movemos todos los parámetros del modelo a la GPU, si está disponible.
network = network.to(device)

network

In [ ]:
# CrossEntropyLoss combina internamente:
# 1. LogSoftmax
# 2. Negative Log-Likelihood Loss
#
# Por eso la última capa entrega 10 logits y NO agregamos Softmax al modelo.
loss_fn = nn.CrossEntropyLoss()

# Equivalente al optimizador RMSprop utilizado en el notebook original.
optimizer = torch.optim.RMSprop(network.parameters(), lr=1e-3)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))
ax.set_xlim(-0.5, 13)
ax.set_ylim(-1.5, 8)
ax.axis('off')
ax.set_title('Arquitectura de nuestra red neuronal', fontsize=16, fontweight='bold', pad=20)

capas = [
    {'x': 1.5, 'n_mostrar': 7, 'total': 784, 'nombre': 'Entrada\n784 píxeles', 'color': '#AED6F1', 'activacion': None},
    {'x': 5, 'n_mostrar': 6, 'total': 32, 'nombre': 'Capa oculta 1\n32 neuronas', 'color': '#A9DFBF', 'activacion': 'tanh'},
    {'x': 8.5, 'n_mostrar': 5, 'total': 16, 'nombre': 'Capa oculta 2\n16 neuronas', 'color': '#FAD7A0', 'activacion': 'sigmoid'},
    {'x': 12, 'n_mostrar': 5, 'total': 10, 'nombre': 'Salida\n10 logits', 'color': '#F5B7B1', 'activacion': 'softmax al predecir'},
]

posiciones_y = {}
for capa in capas:
    ys = np.linspace(1.5, 6.5, capa['n_mostrar'])
    posiciones_y[capa['x']] = ys

for i in range(len(capas) - 1):
    x1, x2 = capas[i]['x'], capas[i + 1]['x']
    for y1 in posiciones_y[x1]:
        for y2 in posiciones_y[x2]:
            ax.plot([x1, x2], [y1, y2], color='gray', alpha=0.08, linewidth=0.5, zorder=1)

for capa in capas:
    ys = posiciones_y[capa['x']]
    for y in ys:
        circulo = plt.Circle((capa['x'], y), 0.22, color=capa['color'],
                              ec='#2C3E50', linewidth=1.5, zorder=3)
        ax.add_patch(circulo)

    if capa['total'] > capa['n_mostrar']:
        ax.text(capa['x'], 0.8, '⋮', ha='center', va='center', fontsize=18, color='#555')

    ax.text(capa['x'], -0.3, capa['nombre'], ha='center', va='top',
            fontsize=10, fontweight='bold', color='#2C3E50')

    if capa['activacion']:
        ax.text(capa['x'], 7.2, f"Activación: {capa['activacion']}",
                ha='center', va='bottom', fontsize=9, style='italic', color='#7F8C8D',
                bbox=dict(boxstyle='round,pad=0.3', facecolor=capa['color'], alpha=0.5))

for i in range(len(capas) - 1):
    mid_x = (capas[i]['x'] + capas[i + 1]['x']) / 2
    ax.annotate('', xy=(mid_x + 0.3, 4), xytext=(mid_x - 0.3, 4),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2))

plt.tight_layout()
plt.show()

Cada capa **transforma** los datos. La primera recibe los 784 píxeles de la imagen y los comprime a 32 valores. La segunda los comprime aún más a 16. La última produce **10 logits**, uno por cada dígito (0-9).

Durante el entrenamiento, `CrossEntropyLoss` convierte internamente estos logits en valores apropiados para calcular el error. Al hacer predicciones podemos aplicar `softmax` para obtener probabilidades; el dígito con la probabilidad más alta es la predicción de la red.

## Preparando los datos y entrenando

Antes de entrenar, necesitamos preparar los datos: convertir cada imagen de una cuadrícula 28×28 a una lista de 784 números, y **normalizar** los valores de los píxeles para que estén entre 0 y 1 (en vez de 0 a 255).

In [ ]:
ejemplo_norm = train_images[0].copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

im1 = ax1.imshow(ejemplo_norm, cmap='gray', vmin=0, vmax=255)
ax1.set_title('Antes: valores de 0 a 255', fontsize=12)
ax1.axis('off')
plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04, label='Valor del píxel')

ejemplo_normalizado = ejemplo_norm.astype('float32') / 255
im2 = ax2.imshow(ejemplo_normalizado, cmap='gray', vmin=0, vmax=1)
ax2.set_title('Después: valores de 0.0 a 1.0', fontsize=12)
ax2.axis('off')
plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04, label='Valor del píxel')

fig.suptitle('¿Por qué normalizamos? Misma imagen, diferente escala',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Convertimos cada imagen de 28×28 a un vector de 784 valores.
# Después normalizamos los píxeles para que estén entre 0 y 1.
train_images = train_images.reshape((60000, 28 * 28)).astype("float32") / 255.0
test_images = test_images.reshape((10000, 28 * 28)).astype("float32") / 255.0

# Convertimos los arreglos de NumPy a tensores de PyTorch.
X_train = torch.from_numpy(train_images)
y_train = torch.from_numpy(train_labels).long()

X_test = torch.from_numpy(test_images)
y_test = torch.from_numpy(test_labels).long()

# TensorDataset asocia cada imagen con su etiqueta.
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

BATCH_SIZE = 128
use_cuda = device.type == "cuda"

# DataLoader divide los datos en lotes y mezcla el entrenamiento.
# pin_memory acelera la transferencia CPU → GPU cuando usamos CUDA.
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=use_cuda
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=2,
    pin_memory=use_cuda
)

In [ ]:
# En Keras se usaban etiquetas one-hot.
# CrossEntropyLoss de PyTorch espera directamente enteros entre 0 y 9,
# así que no necesitamos convertir las etiquetas con to_categorical.

imagenes_batch, etiquetas_batch = next(iter(train_loader))

print("Forma de un lote de imágenes:", imagenes_batch.shape)
print("Forma de un lote de etiquetas:", etiquetas_batch.shape)
print("Primeras etiquetas:", etiquetas_batch[:10])

In [ ]:
history = {
    "loss": [],
    "accuracy": []
}

EPOCHS = 5

for epoch in range(EPOCHS):
    # Activa el modo de entrenamiento.
    network.train()

    running_loss = 0.0
    running_correct = 0
    total_examples = 0

    for batch_images, batch_labels in train_loader:
        # Enviamos cada lote a la GPU.
        batch_images = batch_images.to(device, non_blocking=use_cuda)
        batch_labels = batch_labels.to(device, non_blocking=use_cuda)

        # Reiniciamos los gradientes de la iteración anterior.
        optimizer.zero_grad(set_to_none=True)

        # Forward pass: calculamos los logits.
        logits = network(batch_images)

        # Calculamos el error.
        loss = loss_fn(logits, batch_labels)

        # Backpropagation: calculamos gradientes.
        loss.backward()

        # Actualizamos los pesos.
        optimizer.step()

        batch_size_actual = batch_labels.size(0)
        running_loss += loss.item() * batch_size_actual
        running_correct += (logits.argmax(dim=1) == batch_labels).sum().item()
        total_examples += batch_size_actual

    epoch_loss = running_loss / total_examples
    epoch_accuracy = running_correct / total_examples

    history["loss"].append(epoch_loss)
    history["accuracy"].append(epoch_accuracy)

    print(
        f"Época {epoch + 1}/{EPOCHS} - "
        f"loss: {epoch_loss:.4f} - accuracy: {epoch_accuracy:.4f}"
    )

In [ ]:
# En PyTorch se recomienda guardar los parámetros del modelo (state_dict).
torch.save(
    {
        "model_state_dict": network.state_dict(),
        "input_size": 28 * 28,
        "hidden_sizes": [32, 16],
        "num_classes": 10
    },
    "numbers_pytorch.pth"
)

print("Modelo guardado como numbers_pytorch.pth")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history["accuracy"]) + 1)

ax1.plot(epochs_range, history["accuracy"], 'o-', color='#2ecc71',
         linewidth=2, markersize=8, label='Accuracy')
for i, val in enumerate(history["accuracy"]):
    ax1.annotate(f'{val:.3f}', (i + 1, val), textcoords="offset points",
                 xytext=(0, 10), ha='center', fontsize=9)
ax1.set_xlabel('Época', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_title('¿Qué tan bien aprende? (Accuracy)', fontsize=13, fontweight='bold')
ax1.set_ylim(0.5, 1.0)
ax1.legend(fontsize=11)

ax2.plot(epochs_range, history["loss"], 'o-', color='#e74c3c',
         linewidth=2, markersize=8, label='Loss')
for i, val in enumerate(history["loss"]):
    ax2.annotate(f'{val:.3f}', (i + 1, val), textcoords="offset points",
                 xytext=(0, 10), ha='center', fontsize=9)
ax2.set_xlabel('Época', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.set_title('¿Cuánto se equivoca? (Loss)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()

## ¿Qué tan bien aprendió?

Ya entrenamos la red. Ahora veamos qué tan bien funciona con imágenes que **nunca ha visto** (el conjunto de prueba). Esto nos dice si realmente aprendió a reconocer dígitos, o solo memorizó los ejemplos de entrenamiento.

In [ ]:
# Evaluación: no calculamos gradientes y no actualizamos pesos.
network.eval()

running_test_loss = 0.0
running_test_correct = 0
total_test_examples = 0

with torch.inference_mode():
    for batch_images, batch_labels in test_loader:
        batch_images = batch_images.to(device, non_blocking=use_cuda)
        batch_labels = batch_labels.to(device, non_blocking=use_cuda)

        logits = network(batch_images)
        loss = loss_fn(logits, batch_labels)

        batch_size_actual = batch_labels.size(0)
        running_test_loss += loss.item() * batch_size_actual
        running_test_correct += (logits.argmax(dim=1) == batch_labels).sum().item()
        total_test_examples += batch_size_actual

test_loss = running_test_loss / total_test_examples
test_acc = running_test_correct / total_test_examples

In [ ]:
print(f"test_loss: {test_loss:.4f}")
print(f"test_accuracy: {test_acc:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix

network.eval()

probabilidades_lista = []
predicciones_lista = []
etiquetas_reales_lista = []

with torch.inference_mode():
    for batch_images, batch_labels in test_loader:
        batch_images = batch_images.to(device, non_blocking=use_cuda)

        logits = network(batch_images)
        probabilidades = torch.softmax(logits, dim=1)
        etiquetas_pred_batch = probabilidades.argmax(dim=1)

        probabilidades_lista.append(probabilidades.cpu())
        predicciones_lista.append(etiquetas_pred_batch.cpu())
        etiquetas_reales_lista.append(batch_labels)

# Unimos todos los lotes y regresamos a NumPy para usar sklearn.
predicciones = torch.cat(probabilidades_lista).numpy()
etiquetas_pred = torch.cat(predicciones_lista).numpy()
etiquetas_real = torch.cat(etiquetas_reales_lista).numpy()

cm = confusion_matrix(etiquetas_real, etiquetas_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=range(10), yticklabels=range(10),
            linewidths=0.5, linecolor='white')
ax.set_xlabel('Lo que predijo la red', fontsize=12)
ax.set_ylabel('El dígito real', fontsize=12)
ax.set_title('Matriz de confusión: ¿qué dígitos confunde?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
correctas = np.where(etiquetas_pred == etiquetas_real)[0]
incorrectas = np.where(etiquetas_pred != etiquetas_real)[0]

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle('Predicciones de la red', fontsize=14, fontweight='bold', y=1.02)

axes[0, 0].set_ylabel('Aciertos ✓', fontsize=12, color='green', fontweight='bold')
for i in range(8):
    idx = correctas[i]
    img = test_images[idx].reshape(28, 28)
    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].set_title(f'{etiquetas_pred[idx]}', fontsize=12, color='green', fontweight='bold')
    axes[0, i].axis('off')

axes[1, 0].set_ylabel('Errores ✗', fontsize=12, color='red', fontweight='bold')
for i in range(8):
    idx = incorrectas[i]
    img = test_images[idx].reshape(28, 28)
    axes[1, i].imshow(img, cmap='gray')
    axes[1, i].set_title(f'Pred: {etiquetas_pred[idx]}  Real: {etiquetas_real[idx]}',
                          fontsize=9, color='red', fontweight='bold')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()